<a href="https://colab.research.google.com/github/jainlavanya14/Citation_Intent_Classification/blob/main/Scibertmodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets scikit-learn -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("kejian/MultiCite-classification-gold-context")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/737 [00:00<?, ?B/s]

data/train-00000-of-00001-4cea01d3333d8b(…):   0%|          | 0.00/716k [00:00<?, ?B/s]

data/test-00000-of-00001-893affe0c9c6324(…):   0%|          | 0.00/433k [00:00<?, ?B/s]

data/validation-00000-of-00001-3fabce670(…):   0%|          | 0.00/329k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5491 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3313 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2447 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'x', 'y'],
        num_rows: 5491
    })
    test: Dataset({
        features: ['id', 'x', 'y'],
        num_rows: 3313
    })
    validation: Dataset({
        features: ['id', 'x', 'y'],
        num_rows: 2447
    })
})

In [ ]:

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'x', 'y'],
        num_rows: 5491
    })
    test: Dataset({
        features: ['id', 'x', 'y'],
        num_rows: 3313
    })
    validation: Dataset({
        features: ['id', 'x', 'y'],
        num_rows: 2447
    })
})

In [ ]:
import pandas as pd

train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

In [ ]:
train_df = train_df.rename(columns={"x":"text","y":"labels"})
val_df = val_df.rename(columns={"x":"text","y":"labels"})
test_df = test_df.rename(columns={"x":"text","y":"labels"})

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

train_labels = train_df["labels"].apply(lambda x: x.split())
val_labels = val_df["labels"].apply(lambda x: x.split())
test_labels = test_df["labels"].apply(lambda x: x.split())

mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(train_labels)
y_val = mlb.transform(val_labels)
y_test = mlb.transform(test_labels)

num_labels = len(mlb.classes_)

print(mlb.classes_)

['background' 'differences' 'extends' 'future_work' 'motivation'
 'similarities' 'uses']


In [ ]:
from datasets import Dataset

train_hf = Dataset.from_dict({
    "text": train_df["text"].tolist(),
    "labels": y_train.tolist()
})

val_hf = Dataset.from_dict({
    "text": val_df["text"].tolist(),
    "labels": y_val.tolist()
})

test_hf = Dataset.from_dict({
    "text": test_df["text"].tolist(),
    "labels": y_test.tolist()
})

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "allenai/scibert_scivocab_uncased"
)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

train_hf = train_hf.map(tokenize, batched=True)
val_hf = val_hf.map(tokenize, batched=True)
test_hf = test_hf.map(tokenize, batched=True)

Map:   0%|          | 0/5491 [00:00<?, ? examples/s]

Map:   0%|          | 0/2447 [00:00<?, ? examples/s]

Map:   0%|          | 0/3313 [00:00<?, ? examples/s]

In [ ]:
train_hf.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
val_hf.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
test_hf.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

In [ ]:
import torch

def data_collator(features):

    input_ids = torch.stack([f["input_ids"] for f in features])
    attention_mask = torch.stack([f["attention_mask"] for f in features])

    # stack label tensors and convert to float
    labels = torch.stack([f["labels"] for f in features]).float()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "allenai/scibert_scivocab_uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = 1/(1+np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    macro_f1 = f1_score(labels, preds, average="macro")
    micro_f1 = f1_score(labels, preds, average="micro")

    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="scibert_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    learning_rate=2e-5
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.129005
1000,0.105173
1500,0.091645
2000,0.068676
2500,0.048119


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2748, training_loss=0.08507912370662328, metrics={'train_runtime': 2555.5801, 'train_samples_per_second': 8.595, 'train_steps_per_second': 1.075, 'total_flos': 5779230655180800.0, 'train_loss': 0.08507912370662328, 'epoch': 4.0})

In [ ]:
trainer.evaluate(test_hf)

{'eval_loss': 0.2984440326690674,
 'eval_macro_f1': 0.6198518377435284,
 'eval_micro_f1': 0.7214428857715431,
 'eval_runtime': 98.4732,
 'eval_samples_per_second': 33.644,
 'eval_steps_per_second': 4.214,
 'epoch': 4.0}